In [1]:
from urllib.parse import urlparse
import pickle
import pandas as pd
import psycopg2
from psycopg2 import pool
from concurrent.futures import ThreadPoolExecutor, as_completed
pd.set_option('display.max_columns', None)
#pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

# Data Extraction

In [2]:
connection_pool = psycopg2.pool.ThreadedConnectionPool(
    minconn=1,
    maxconn=100,  # Adjust based on your PostgreSQL max_connections and credentials
    host="localhost",
    database="altered_histories",
    user="rapaport",
    password="WHT6rNYJknNme3lS"
)

In [3]:
def execute_query_with_params(query, params):
    conn = connection_pool.getconn()
    try:
        with conn.cursor() as cursor:
            cursor.execute("SET max_parallel_workers = 24;")
            cursor.execute("SET max_parallel_workers_per_gather = 4;")
            
            # Execute your actual query
            cursor.execute(query, params)
            return cursor.fetchall()
    finally:
        connection_pool.putconn(conn)

In [4]:
secret_patterns = [
    '%/.ssh%',
    '%/id_dsa',
    '%/id_ecdsa', 
    '%/id_rsa',
    '%secret%',
    '%/.env%',
    '%credential%',
    '%/secring.gpg',
    '%/.npmrc',
    '%/.pypirc',
    '%/.netrc',
    '%key%',
    '%.pem',
    '%.pfx',
    '%.p12',
    '%.jks'
]
base_query = "SELECT * FROM modified_files WHERE path LIKE %s"

In [5]:
with ThreadPoolExecutor(max_workers=6) as executor:
    futures = [executor.submit(execute_query_with_params, base_query, (pattern,)) 
               for pattern in secret_patterns]
    
    all_results = []
    for future in as_completed(futures):
        results = future.result()
        all_results.extend(results)
        print(f"Pattern found {len(results)} matches")

Pattern found 59 matches
Pattern found 137718 matches
Pattern found 1809860 matches
Pattern found 2168 matches
Pattern found 232 matches
Pattern found 759 matches
Pattern found 60 matches
Pattern found 1797966 matches
Pattern found 572 matches
Pattern found 18976 matches
Pattern found 26 matches
Pattern found 442498 matches
Pattern found 17307738 matches
Pattern found 10514 matches
Pattern found 19440 matches
Pattern found 65764 matches


In [6]:
columns = ['id', 'origin', 'revision_id', 'branch', 'snapshot_id', 'path', 'status']
df_all_results = pd.DataFrame(all_results, columns=columns)
print(f"Total rows: {len(df_all_results):,}")
df_all_results.head()

Total rows: 21,614,350


,id,origin,revision_id,branch,snapshot_id,path,status
0,1264069815,https://github.com/VikramChilkunda/tutormatching,swh:1:rev:0093b476c23be49e322010c3edabe5a18d872fe5,refs/heads/dependabot/npm_and_yarn/heroku/tar-4.4.19,swh:1:snp:7f47cbdf4e6174d5834cfb44228158b014f0836c,./heroku/node_modules/tunnel-ssh/node_modules/ssh2/test/fixtures/id_ecdsa,NotFound
1,1264101734,https://github.com/VikramChilkunda/tutormatching,swh:1:rev:34487d7752b8e27cedddf9aec32e12454583e1c7,refs/pull/51/head,swh:1:snp:7f47cbdf4e6174d5834cfb44228158b014f0836c,./heroku/node_modules/tunnel-ssh/node_modules/ssh2/test/fixtures/id_ecdsa,NotFound
2,1264293630,https://github.com/VikramChilkunda/tutormatching,swh:1:rev:c4140b0818fc3f1d3cca2a3e2f28c958899a0262,refs/pull/36/head,swh:1:snp:7f47cbdf4e6174d5834cfb44228158b014f0836c,./heroku/node_modules/tunnel-ssh/node_modules/ssh2/test/fixtures/id_ecdsa,NotFound
3,1264336502,https://github.com/VikramChilkunda/tutormatching,swh:1:rev:0093b476c23be49e322010c3edabe5a18d872fe5,refs/pull/48/head,swh:1:snp:7f47cbdf4e6174d5834cfb44228158b014f0836c,./heroku/node_modules/tunnel-ssh/node_modules/ssh2/test/fixtures/id_ecdsa,NotFound
4,1344840927,https://github.com/dDamyanov0305/ora.jenkins,swh:1:rev:5003e2a6b38964d65c9a34347b7b67df39be4e4a,refs/pull/1/head,swh:1:snp:a7f5674ab6ee8a769bb247a6d96bd840fe9e5f99,./backend/node_modules/ssh2/test/fixtures/id_ecdsa,NotFound


In [7]:
with open("../../data/secret_files_full.pkl", "wb") as f:
    pickle.dump(df_all_results, f)

# Data analysis

In [ ]:
df = pd.read_pickle("../../data/secret_files_full.pkl")

In [ ]:
df.head(2)

,id,origin,revision_id,branch,snapshot_id,path,status
0,126773317,https://github.com/mergar/cbsd,swh:1:rev:663c93539dd8ad04248362ae842d9681cc8fe898,HEAD,swh:1:snp:564048f873fe72e9217e281e03230c23ec9e4ede,./tmp/share/FreeBSD-jail-webdev-skel/root/jails/.ssh/db00721f6467d291e49c4a9eeb231f11.id_rsa,NotFound
1,122583025,https://github.com/jaymzjulian/chef,swh:1:rev:4dd976e7167e288830f76a107a0b71147efde6ce,refs/heads/jj/faster-chocolatey,swh:1:snp:80e6f78440c40188a541f87adf9b02eb9c54aed6,./spec/data/cookbooks/openldap/files/default/.ssh/id_rsa,NotFound


We remove the alias `HEAD` which produces duplicates

In [ ]:
print(f'amount of secret files found (including HEAD): {len(df):,}')
df = df[df['branch'] != "HEAD"]
print(f'Amount of secret files found (excluding HEAD): {len(df):,}')

amount of secret files found (including HEAD): 21,614,350
Amount of secret files found (excluding HEAD): 21,109,512


In [ ]:
print(f'amount of secret files found: {len(df):,}')
df_removed = df[df['status']=="NotFound"]
df_modified = df[df['status']=="Modified"]
print(f'amount of files removed: {len(df_removed):,}') 
from gen_tex import update_latex_macros
print(f'amount of files modified: {len(df_modified):,}')

amount of secret files found: 21,109,512
amount of files removed: 8,055,854
amount of files modified: 13,053,658


True

In [ ]:
secrets_origins = df['origin'].drop_duplicates().reset_index(drop=True)
print(f'amount of origin matching the secret pattern: {len(secrets_origins):,}')

amount of origin matching the secret pattern: 75,054


True

# Cross results with stars

In [ ]:
stars = pd.read_pickle('../../data/stars_without_dup.pkl')

In [6]:
stars_dict = stars.set_index('origin')['stars'].to_dict()

In [7]:
df['stars'] = df['origin'].map(stars_dict).fillna(0)

# Pattern Analysis by Status

## Focus on removed files

In [8]:
# Define the search patterns (cleaned from SQL wildcards)
patterns = [
    '%/.ssh%',
    '%/id_dsa',
    '%/id_ecdsa', 
    '%/id_rsa',
    '%secret%',
    '%/.env%',
    '%credential%',
    '%/secring.gpg',
    '%/.npmrc',
    '%/.pypirc',
    '%/.netrc',
    '%key%',
    '%.pem',
    '%.pfx',
    '%.p12',
    '%.jks'
]

# Create a comprehensive analysis
pattern_analysis = []

for pattern in patterns:
    # Use simple contains operation for all patterns
    matches = df_removed[df_removed['path'].str.contains(pattern.strip('%'), case=False, na=False)]
    
    total_count = len(matches)
    
    pattern_analysis.append({
        'Pattern': pattern,
        'Total': total_count,
    })

# Create DataFrame for analysis
pattern_df = pd.DataFrame(pattern_analysis)
pattern_df = pattern_df.sort_values('Total', ascending=False)

print("Secret File Pattern Analysis:")
print("=" * 50)
print(pattern_df.to_string(index=False))

Secret File Pattern Analysis:
      Pattern   Total
        %key% 6910188
     %secret%  805432
 %credential%  545591
        %.pem  357501
        %.jks  101139
      %/.env%   43868
        %.p12   21448
     %/.npmrc   11633
        %.pfx    7763
      %/.ssh%    6387
     %/id_rsa    1249
%/secring.gpg     691
     %/id_dsa     222
   %/id_ecdsa      75
    %/.pypirc      42
     %/.netrc      16


In [ ]:
df_removed[df['removed'].str.contains('key')]

In [ ]:
df_removed[df['removed'].str.contains('secret')]

In [ ]:
df_removed[df['removed'].str.contains('/id_rsa$')]

In [ ]:
df_removed[(df_removed['branch'] == 'refs/heads/master') | (df_removed['branch'] == 'refs/heads/main')]

## Popular Repositories (1000+)

In [ ]:
df_1000 = df[df['stars']>= 1000]
print(f'amount of secrets in 1000+ stars origins: {len(df_1000):,}')

amount of secrets in 1000+ stars origins: 3,919,794


True

In [ ]:
secrets_origins_1000 = df_1000['origin'].drop_duplicates().reset_index(drop=True)
print(f'amount of origins with 1000+ stars containing secrets: {len(secrets_origins_1000):,}')

amount of origins with 1000+ stars containing secrets: 4,562


True

In [13]:
pattern_analysis = []

for pattern in patterns:
    # Use simple contains operation for all patterns
    matches = df_1000[df_1000['path'].str.contains(pattern.strip('%'), case=False, na=False)]
    
    total_count = len(matches)
    
    pattern_analysis.append({
        'Pattern': pattern,
        'Total': total_count,
    })

# Create DataFrame for analysis
pattern_df = pd.DataFrame(pattern_analysis)
pattern_df = pattern_df.sort_values('Total', ascending=False)

print("Secret File Pattern Analysis:")
print("=" * 50)
print(pattern_df.to_string(index=False))

Secret File Pattern Analysis:
      Pattern   Total
        %key% 2979135
 %credential%  483610
     %secret%  429123
        %.pem  207905
        %.jks   49627
      %/.env%   24344
        %.p12   10054
     %/.npmrc    8821
        %.pfx    1689
      %/.ssh%    1555
     %/id_rsa     437
%/secring.gpg     369
     %/id_dsa      59
    %/.pypirc      12
   %/id_ecdsa       7
     %/.netrc       0


In [ ]:
id_rsa = df_1000[df_1000['path'].str.contains('/id_rsa$')]

,id,origin,revision_id,branch,snapshot_id,path,status,stars
534,849093654,https://github.com/chef/chef,swh:1:rev:373f34cb0d85fc9061b959820965046ab7b5332a,refs/pull/13770/head,swh:1:snp:c8e9a5e938051b9a6d944e051b170e46c402e3dc,./knife/spec/data/cookbooks/openldap/files/default/.ssh/id_rsa,NotFound,7034.0
535,848587663,https://github.com/chef/chef,swh:1:rev:4a7f8166916d5f90087f03518deeb5873ed4d07a,refs/pull/14283/head,swh:1:snp:f51b6575e99c6218f9fcf526bfa0fa521f273ad2,./knife/spec/data/cookbooks/openldap/files/default/.ssh/id_rsa,NotFound,7034.0
536,848968070,https://github.com/chef/chef,swh:1:rev:72127f088446f720cf631a71f22c1a89175449f4,refs/pull/13249/head,swh:1:snp:07247905c4e7458575bf82da605d7bafec436140,./spec/data/cookbooks/openldap/files/default/.ssh/id_rsa,NotFound,7034.0
537,848809911,https://github.com/chef/chef,swh:1:rev:8dd9d6656a7726c1429f9e9e07b99dd8f428b10f,refs/pull/14052/head,swh:1:snp:10eb22b9cdc98a4e0dd7f05a61114b57fb38318e,./knife/spec/data/cookbooks/openldap/files/default/.ssh/id_rsa,NotFound,7034.0
540,848757789,https://github.com/chef/chef,swh:1:rev:fff8dfc1b20f364a0a1fc91473f7090114bd463d,refs/pull/11626/head,swh:1:snp:c4f76d28696b93a0257c3bf730ef84f7d2de5ab3,./spec/data/cookbooks/openldap/files/default/.ssh/id_rsa,NotFound,7034.0
...,...,...,...,...,...,...,...,...
1950721,1733008362,https://github.com/sshnet/SSH.NET,swh:1:rev:186aec3d3c7b3d0a6c51c6083d9452c0f33c9c6d,refs/pull/1234/head,swh:1:snp:878cb343b272b405e02d0ef31cf7914b7d088360,./test/Renci.SshNet.IntegrationTests/resources/client/id_rsa,NotFound,4132.0
4847591,1139737306,https://github.com/docker/docker-py,swh:1:rev:78f3693bbf0b65c1df63951adbfb5303c5a259e6,refs/pull/2478/head,swh:1:snp:87f1e6be2216e31b02d77308a7aaa5dbd3906251,./tests/ssh-keys/id_rsa,NotFound,5967.0
4848666,1139977235,https://github.com/docker/docker-py,swh:1:rev:797ccb84e591a7a0dd7ed3b2dad6d032d540482e,refs/pull/2865/head,swh:1:snp:6ade7ed7217b251ab4861f55e2ba0f9e9b2facf1,./tests/ssh-keys/id_rsa,NotFound,5967.0
4850880,1140382794,https://github.com/docker/docker-py,swh:1:rev:068d6c0270dcabbe7229d65cd63ba7ba2ea01f15,refs/pull/2803/head,swh:1:snp:0f5f7ea6bc544651cb14008a0c2937fbfbbb5c44,./tests/ssh-keys/id_rsa,NotFound,5967.0


In [21]:
df_1000[df_1000['branch'].str.contains('pull')]

,id,origin,revision_id,branch,snapshot_id,path,status,stars
281,504920523,https://github.com/materialsproject/pymatgen,swh:1:rev:6c54a5fc63bec2151fc4990a2dddf10e1ddadc99,refs/pull/2372/head,swh:1:snp:b3753f7d3769a632d40bb6223735624ca5ea0b25,./test_files/xtb/sample_CREST_output/.ssh/config,NotFound,1625.0
283,504920384,https://github.com/materialsproject/pymatgen,swh:1:rev:6c54a5fc63bec2151fc4990a2dddf10e1ddadc99,refs/pull/2372/head,swh:1:snp:b3753f7d3769a632d40bb6223735624ca5ea0b25,./test_files/xtb/sample_CREST_output/.ssh/known_hosts,NotFound,1625.0
284,504920499,https://github.com/materialsproject/pymatgen,swh:1:rev:6c54a5fc63bec2151fc4990a2dddf10e1ddadc99,refs/pull/2372/head,swh:1:snp:b3753f7d3769a632d40bb6223735624ca5ea0b25,./test_files/xtb/sample_CREST_output/.ssh/perceus.pub,NotFound,1625.0
285,504921233,https://github.com/materialsproject/pymatgen,swh:1:rev:6c54a5fc63bec2151fc4990a2dddf10e1ddadc99,refs/pull/2372/head,swh:1:snp:b3753f7d3769a632d40bb6223735624ca5ea0b25,./test_files/xtb/sample_CREST_output/.ssh/authorized_keys,NotFound,1625.0
288,504921211,https://github.com/materialsproject/pymatgen,swh:1:rev:6c54a5fc63bec2151fc4990a2dddf10e1ddadc99,refs/pull/2372/head,swh:1:snp:b3753f7d3769a632d40bb6223735624ca5ea0b25,./test_files/xtb/sample_CREST_output/.ssh/perceus,NotFound,1625.0
...,...,...,...,...,...,...,...,...
21614321,383951855,https://github.com/keycloak/keycloak,swh:1:rev:590020a6b5040e44962b063eb6c36b64b34f2a1c,refs/pull/28597/head,swh:1:snp:637746ffd8d4f5d9dd496d2370c59247c6069f42,./testsuite/integration-arquillian/tests/base/src/test/resources/adapter-test/secure-portal-with-custom-session-config/WEB-INF/keystore.jks,NotFound,25283.0
21614322,383952012,https://github.com/keycloak/keycloak,swh:1:rev:590020a6b5040e44962b063eb6c36b64b34f2a1c,refs/pull/28597/head,swh:1:snp:637746ffd8d4f5d9dd496d2370c59247c6069f42,./testsuite/integration-arquillian/servers/app-server/tomcat/common/common-files/keystore/adapter.jks,NotFound,25283.0
21614323,383952072,https://github.com/keycloak/keycloak,swh:1:rev:590020a6b5040e44962b063eb6c36b64b34f2a1c,refs/pull/28597/head,swh:1:snp:637746ffd8d4f5d9dd496d2370c59247c6069f42,./testsuite/integration-arquillian/tests/base/src/test/resources/adapter-test/keycloak-saml/employee-sig-post-noidpkey/WEB-INF/keystore.jks,Modified,25283.0
21614348,384546352,https://github.com/OpenLiberty/open-liberty,swh:1:rev:c2a94d1ee57e8374ae1085d3cdaf33984503618f,refs/pull/25465/head,swh:1:snp:5cd26fa35eb4ddaffa3de793247a971feae5c49b,./dev/io.openliberty.checkpoint_fat_transaction/publish/servers/checkpointTransactionServlet/resources/security/defaultTrust.jks,NotFound,1017.0
